In [ ]:
    @Override
    public Promise<Map<SuggestionStatus, Long>> countByStatus(String tenantId) {
        return Promise.ofBlocking(executor, () -> {
            String sql = "SELECT status, COUNT(*) as cnt FROM " + TABLE + " WHERE tenant_id = ? GROUP BY status";
            Map<SuggestionStatus, Long> result = new EnumMap<>(SuggestionStatus.class);
            // Initialize with 0
            for (SuggestionStatus status : SuggestionStatus.values()) {
                result.put(status, 0L);
            }
            try (Connection conn = dataSource.getConnection();
                 PreparedStatement ps = conn.prepareStatement(sql)) {
                ps.setString(1, tenantId);
                try (ResultSet rs = ps.executeQuery()) {
                    while (rs.next()) {
                        String statusStr = rs.getString("status");
                        if (statusStr != null) {
                            try {
                                result.put(SuggestionStatus.valueOf(statusStr), rs.getLong("cnt"));
                            } catch (IllegalArgumentException ignored) {}
                        }
                    }
                }
            }
            return result;
        });
    }

    @Override
    public Promise<Map<SuggestionType, Long>> countPendingByType(String tenantId) {
        return Promise.ofBlocking(executor, () -> {
            String sql = "SELECT type, COUNT(*) as cnt FROM " + TABLE + " WHERE tenant_id = ? AND status = 'PENDING' GROUP BY type";
            Map<SuggestionType, Long> result = new EnumMap<>(SuggestionType.class);
            for (SuggestionType type : SuggestionType.values()) {
                result.put(type, 0L);
            }
            try (Connection conn = dataSource.getConnection();
                 PreparedStatement ps = conn.prepareStatement(sql)) {
                ps.setString(1, tenantId);
                try (ResultSet rs = ps.executeQuery()) {
                    while (rs.next()) {
                         String typeStr = rs.getString("type");
                        if (typeStr != null) {
                            try {
                                result.put(SuggestionType.valueOf(typeStr), rs.getLong("cnt"));
                            } catch (IllegalArgumentException ignored) {}
                        }
                    }
                }
            }
            return result;
        });
    }

    @Override
    public Promise<Double> getAcceptanceRate(String tenantId) {
        return Promise.ofBlocking(executor, () -> {
            String sql = "SELECT " +
                         "SUM(CASE WHEN status = 'ACCEPTED' THEN 1 ELSE 0 END) as accepted, " +
                         "SUM(CASE WHEN status = 'REJECTED' THEN 1 ELSE 0 END) as rejected " +
                         "FROM " + TABLE + " WHERE tenant_id = ?";
            try (Connection conn = dataSource.getConnection();
                 PreparedStatement ps = conn.prepareStatement(sql)) {
                ps.setString(1, tenantId);
                try (ResultSet rs = ps.executeQuery()) {
                    if (rs.next()) {
                        long accepted = rs.getLong("accepted");
                        long rejected = rs.getLong("rejected");
                        long total = accepted + rejected;
                        if (total == 0) return 0.0;
                        return (double) accepted / total;
                    }
                }
            }
            return 0.0;
        });
    }